# 访问历史模型：临床与组学访问并集

在训练阶段用临床访问和组学访问构建历史，在预测阶段按官方 API 的 `test`、`test_peptides`、`test_proteins` 批次累积已观察访问。保持比赛数据源挂载并关闭 Internet。

In [ ]:
import sys
sys.path.append('/kaggle/input/amp-pd')

In [ ]:
"""Time baseline and causal visit-history model for the AMP competition.

Notebook: model = VisitHistoryModel.from_data_dir('/kaggle/input/amp-pd')
          predictions = model.predict_batch(test, test_peptides, test_proteins, sample)
Call predict_batch once per API batch, in the order supplied by iter_test().
Local validation: python C_improved_model/visit_history_model.py --validate
"""
from pathlib import Path
import argparse
import json

import numpy as np
import pandas as pd

TARGETS = [f'updrs_{i}' for i in range(1, 5)]
KEYS = ['patient_id', 'visit_month']
MILESTONES = (6, 18)


def read_data(data_dir):
    data_dir = Path(data_dir)
    clinical = pd.read_csv(data_dir / 'train_clinical_data.csv')
    omics = pd.concat([pd.read_csv(data_dir / name, usecols=KEYS) for name in
                       ('train_proteins.csv', 'train_peptides.csv')]).drop_duplicates()
    return clinical, omics


def history_codes(requests, visits):
    """Milestone visits are visible only at or after their month."""
    result = np.zeros(len(requests), dtype=int)
    for i, milestone in enumerate(MILESTONES):
        seen = set(visits.loc[visits.visit_month.eq(milestone), 'patient_id'])
        digit = np.where(requests.visit_month.to_numpy() < milestone, -1,
                         requests.patient_id.isin(seen).to_numpy().astype(int))
        result += (digit + 1) * 3 ** i
    return result


def training_rows(clinical, omics, stream='union'):
    origins = omics.drop_duplicates().copy()
    requests = origins.merge(pd.DataFrame({'horizon': [0, 6, 12, 24]}), how='cross')
    requests['target_month'] = requests.visit_month + requests.horizon
    labels = clinical.melt(id_vars=KEYS, value_vars=TARGETS, var_name='target',
                           value_name='rating').dropna(subset=['rating'])
    labels = labels.rename(columns={'visit_month': 'target_month'})
    rows = requests.merge(labels, on=['patient_id', 'target_month'], validate='many_to_many')
    if rows.duplicated(['patient_id', 'visit_month', 'target', 'horizon']).any():
        raise ValueError('Duplicate training requests; check clinical visit keys')
    visits = pd.concat([clinical[KEYS], omics]).drop_duplicates() if stream == 'union' else omics
    rows['history'] = history_codes(rows, visits)
    return rows


class VisitHistoryModel:
    def __init__(self, rows, use_history=True):
        self.rows = rows.copy()
        self.use_history = use_history
        self.visits = set()
        self.last_month = None

    @classmethod
    def from_data_dir(cls, data_dir, use_history=True, stream='union'):
        clinical, omics = read_data(data_dir)
        return cls(training_rows(clinical, omics, stream), use_history)

    def predict_rows(self, rows):
        result = pd.Series(index=rows.index, dtype=float)
        groups = ['target', 'target_month'] + (['history'] if self.use_history else [])
        for key, part in rows.groupby(groups, sort=False):
            target, month = key[:2]
            selected = self.rows.loc[self.rows.target.eq(target)]
            if self.use_history:
                same = selected.loc[selected.history.eq(key[2])]
                if not same.empty:
                    selected = same
            truth = selected.rating.to_numpy()[:, None]
            candidates = np.arange(int(truth.max()) + 1)
            losses = 200 * np.abs(truth - candidates) / (truth + candidates + 2)
            weights = np.exp(-0.5 * ((selected.target_month.to_numpy() - month) / 12) ** 2)
            result.loc[part.index] = candidates[(weights @ losses).argmin()]
        return result

    def predict_batch(self, test, peptides, proteins, sample):
        match = sample.prediction_id.str.extract(
            r'^(?P<patient_id>\d+)_(?P<visit_month>\d+)_(?P<target>updrs_[1-4])_plus_(?P<horizon>0|6|12|24)_months$')
        if match.isna().any().any():
            raise ValueError('Unexpected prediction_id format')
        for col in ('patient_id', 'visit_month', 'horizon'):
            match[col] = match[col].astype(int)
        match['target_month'] = match.visit_month + match.horizon
        current = pd.concat([frame[KEYS] for frame in (test, peptides, proteins)], ignore_index=True).drop_duplicates()
        months = pd.concat([current.visit_month, match.visit_month]).dropna().unique()
        if len(months) != 1:
            raise ValueError('Expected one visit month per API batch')
        month = int(months[0])
        if self.last_month is not None and month < self.last_month:
            raise ValueError('API batches must be in chronological order')
        self.last_month = month
        self.visits.update(map(tuple, current[KEYS].itertuples(index=False, name=None)))
        visits = pd.DataFrame(list(self.visits), columns=KEYS)
        match['history'] = history_codes(match, visits)
        output = sample.copy()
        output['rating'] = self.predict_rows(match).to_numpy()
        return output



model = VisitHistoryModel.from_data_dir('/kaggle/input/amp-pd', use_history=True, stream='union')

In [ ]:
import amp_pd_peptide
env = amp_pd_peptide.make_env()
iter_test = env.iter_test()

In [ ]:
for (test, test_peptides, test_proteins, sample_submission) in iter_test:
    prediction = model.predict_batch(test, test_peptides, test_proteins, sample_submission)
    env.predict(prediction)